# V7_B_N05 — Medicines Before Stock-Out

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft using synthetic data. Outputs support authorized review only.

## Decision contract
Support replenishment, redistribution, cold-chain readiness, and procurement planning. Owners: medical stores, pharmacy, immunization, and health-service authorities. The model does not prescribe treatment or substitute procurement controls.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7505);items=pd.DataFrame({'facility':[f'F{i:02d}' for i in range(24)],'product':rng.choice(['Vaccine-A','Antibiotic-B','ORS-C'],24),'stock':rng.integers(20,500,24),'daily_use':rng.uniform(4,28,24),'lead_days':rng.integers(7,46,24),'cold_chain_ok':rng.choice([0,1],24,p=[.12,.88]),'report_age_days':rng.integers(0,12,24)});items.head().round(1)

## Evidence and quality
Stock, consumption, losses, batches, expiry, orders, pipeline, storage, and reporting date must share product identifiers and units. Stale reports trigger verification.

In [2]:
items['quality_pass']=items.report_age_days<=7;items['days_cover']=items.stock/items.daily_use;items['reorder_gap_days']=items.lead_days+7-items.days_cover;print(items[['days_cover','lead_days','report_age_days']].describe().round(1))

       days_cover  lead_days  report_age_days
count        24.0       24.0             24.0
mean         21.6       26.8              5.0
std          18.7       10.8              3.5
min           1.4       11.0              0.0
25%           8.2       16.8              2.8
50%          18.5       26.5              4.5
75%          29.5       34.5              8.0
max          79.9       45.0             11.0


## Transparent stock-out risk
A facility is at risk when cover is below lead time plus safety buffer. Vaccines additionally require viable cold chain.

In [3]:
items['risk']=items.reorder_gap_days.clip(lower=0);items.loc[(items.product=='Vaccine-A')&(items.cold_chain_ok==0),'risk']+=20;items['disposition']=np.where(~items.quality_pass,'ABSTAIN—VERIFY STOCK REPORT',np.where(items.risk>0,'REPLENISHMENT REVIEW','MONITOR'));print(items.sort_values('risk',ascending=False).head(10).round(1).to_string(index=False))

facility      product  stock  daily_use  lead_days  cold_chain_ok  report_age_days  quality_pass  days_cover  reorder_gap_days  risk                 disposition
     F01    Vaccine-A    213       25.1         44              1                9         False         8.5              42.5  42.5 ABSTAIN—VERIFY STOCK REPORT
     F20 Antibiotic-B    164        8.5         45              1                5          True        19.2              32.8  32.8        REPLENISHMENT REVIEW
     F05        ORS-C    319       20.9         39              1                8         False        15.3              30.7  30.7 ABSTAIN—VERIFY STOCK REPORT
     F14    Vaccine-A    245       13.5         34              1                5          True        18.1              22.9  22.9        REPLENISHMENT REVIEW
     F21 Antibiotic-B     75       23.9         19              1                0          True         3.1              22.9  22.9        REPLENISHMENT REVIEW
     F18    Vaccine-A    308      

## Constrained redistribution scenario
Move only from facilities above a protected cover level. This simplified scenario requires product, batch, expiry, route, temperature, and authorization checks.

In [4]:
items['surplus_units']=((items.days_cover-35).clip(lower=0)*items.daily_use).astype(int);items['need_units']=((items.lead_days+7-items.days_cover).clip(lower=0)*items.daily_use).astype(int);summary=items.groupby('product').agg(surplus=('surplus_units','sum'),need=('need_units','sum'));summary['unmet_after_internal']= (summary.need-summary.surplus).clip(lower=0);print(summary.to_string())

              surplus  need  unmet_after_internal
product                                          
Antibiotic-B        0  1903                  1903
ORS-C              75  1410                  1335
Vaccine-A         442  3807                  3365


## Uncertainty scenario
Consumption can surge and lead times can lengthen. Recalculate under stress before committing procurement or redistribution.

In [5]:
items['stress_need']=(((items.lead_days+10)*1.25-items.days_cover).clip(lower=0)*items.daily_use).astype(int);print(items.groupby('product')[['need_units','stress_need']].sum().to_string())

              need_units  stress_need
product                              
Antibiotic-B        1903         3061
ORS-C               1410         2361
Vaccine-A           3807         6131


## Decision product
Provide reason codes, data age, quantity scenarios, cold-chain status, owner, and review state. Do not expose patient-level information.

In [6]:
work=items[items.disposition!='MONITOR'][['facility','product','days_cover','lead_days','cold_chain_ok','report_age_days','need_units','stress_need','disposition']];assert len(work)>0;print(work.head().round(1).to_string(index=False))

facility      product  days_cover  lead_days  cold_chain_ok  report_age_days  need_units  stress_need                 disposition
     F00    Vaccine-A        18.9         22              1                3         206          432        REPLENISHMENT REVIEW
     F01    Vaccine-A         8.5         44              1                9        1065         1479 ABSTAIN—VERIFY STOCK REPORT
     F02        ORS-C        29.3         27              1                3          29          104        REPLENISHMENT REVIEW
     F03 Antibiotic-B        32.3         43              1                0         251          483        REPLENISHMENT REVIEW
     F04        ORS-C         5.8         19              1                8         517          778 ABSTAIN—VERIFY STOCK REPORT


## Exercises
1. Add expiry and batch constraints. 2. Model pipeline orders. 3. Add service-level targets. 4. Explain why low stock does not justify automatic substitution.

## Exact solutions
1. Exclude expired/soon-expiring stock and preserve batch traceability. 2. Add confirmed quantity and expected arrival with uncertainty. 3. Link safety stock to acceptable stock-out risk and criticality. 4. Substitution requires clinical/pharmacy protocols, indications, interactions, dosage forms, authorization, and communication.

In [7]:
assert items.quality_pass.isin([True,False]).all() and work.disposition.str.len().gt(0).all();print('V7_B_N05_COMPLETE_EXECUTION_PASS')

V7_B_N05_COMPLETE_EXECUTION_PASS
